In [38]:
import json
import string
import pandas as pd
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity
from transformers import DistilBertModel, DistilBertTokenizer

In [39]:
DATA_FILE = "dummy_data.json"
similarity_threshold = 0.75

In [40]:
def load_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    return pd.DataFrame(data["CourseOutcome"]), pd.DataFrame(data["ProgramOutcome"])

In [41]:
co_data, po_data = load_data(DATA_FILE)

In [42]:
def preprocess_text(text):
    """Converts text to lowercase and removes punctuation."""
    text = text.lower()
    text = ''.join([char for char in text if char not in string.punctuation])
    return text

In [43]:
co_data['cleaned_course_outcome_description'] = co_data['course_outcome_description'].apply(preprocess_text)
po_data['cleaned_program_outcome_description'] = po_data['program_outcome_description'].apply(preprocess_text)

In [44]:
co_data

,course_outcome_code,course_outcome_description,cleaned_course_outcome_description
0,CO1,Understand the concepts of information systems...,understand the concepts of information systems...
1,CO2,Apply software methodologies and techniques pr...,apply software methodologies and techniques pr...
2,CO3,Determine and mode information system requirem...,determine and mode information system requirem...
3,CO4,Develop and manage an information system project.,develop and manage an information system project


In [45]:
distilbert_tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
distilbert_model = DistilBertModel.from_pretrained('distilbert-base-uncased')

In [46]:
def generate_embeddings(text_list):
    """Generates BERT embeddings for a list of text descriptions."""
    encoded_input = distilbert_tokenizer(text_list, padding=True, truncation=True, return_tensors='pt', max_length=128)
    distilbert_model.to('cpu')
    with torch.no_grad():
        model_output = distilbert_model(**encoded_input)
    # Use the mean of the token embeddings as the sentence embedding
    embeddings = model_output.last_hidden_state.mean(dim=1)
    return embeddings.tolist()

In [47]:
po_data['po_embeddings'] = generate_embeddings(po_data['cleaned_program_outcome_description'].tolist())
co_data['co_embeddings'] = generate_embeddings(co_data['cleaned_course_outcome_description'].tolist())

In [48]:
po_embeddings_array = np.array(po_data['po_embeddings'].tolist())
co_embeddings_array = np.array(co_data['co_embeddings'].tolist())

In [49]:
similarity_matrix = cosine_similarity(po_embeddings_array, co_embeddings_array)

In [50]:
similarity_matrix

array([[0.81441846, 0.88134958, 0.77040065, 0.82851323],
       [0.81470912, 0.91688204, 0.87627233, 0.87411159],
       [0.81213044, 0.87197868, 0.82925426, 0.83151911],
       [0.85758944, 0.87534875, 0.86257606, 0.89502544],
       [0.82721471, 0.87796585, 0.84797605, 0.87381132],
       [0.81061721, 0.84682656, 0.82112133, 0.83945289],
       [0.81550153, 0.8859216 , 0.81579849, 0.84282545],
       [0.75828892, 0.77576759, 0.6755276 , 0.76305662],
       [0.75847749, 0.8605501 , 0.83651608, 0.92328346],
       [0.79089148, 0.77762161, 0.71206602, 0.76526953],
       [0.78908501, 0.83027154, 0.79238454, 0.84396878],
       [0.76783496, 0.81200565, 0.80268332, 0.82432237],
       [0.88181635, 0.76938637, 0.67556691, 0.77601135],
       [0.79567173, 0.76942265, 0.69452845, 0.79322782],
       [0.59766406, 0.72278806, 0.70663612, 0.73806528]])

In [51]:
# Create a DataFrame to store the relationships
relationships_df = pd.DataFrame(index=co_data['course_outcome_code'], columns=po_data['program_outcome_code'])

In [ ]:
for i in range(similarity_matrix.shape[0]): # Repeat through rows (POs)
    for j in range(similarity_matrix.shape[1]): # Repeat through columns (COs)
        po = po_data.loc[i, 'program_outcome_code']
        co = co_data.loc[j, 'course_outcome_code']
        # Check if similarity is above the threshold and store as 1 or 0
        relationships_df.loc[co, po] = 1 if similarity_matrix[i, j] >= similarity_threshold else 0

In [58]:
import datetime
timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
filename = f"co_po_matrix_{timestamp}.json"

matrix_dict = relationships_df.to_dict(orient="index")

with open(filename, "w", encoding="utf-8") as f:
    json.dump(matrix_dict, f, indent=4)

In [56]:
matrix_dict = relationships_df.to_dict(orient="index")

print("\n=== CO–PO Relationship Dictionary ===")
print(matrix_dict)


=== CO–PO Relationship Dictionary ===
{'CO1': {'PO-a': 1, 'PO-b': 1, 'PO-c': 1, 'PO-d': 1, 'PO-e': 1, 'PO-f': 1, 'PO-g': 1, 'PO-h': 1, 'PO-i': 1, 'PO-j': 1, 'PO-k': 1, 'PO-l': 1, 'PO-m': 1, 'PO-n': 1, 'PO-o': 0}, 'CO2': {'PO-a': 1, 'PO-b': 1, 'PO-c': 1, 'PO-d': 1, 'PO-e': 1, 'PO-f': 1, 'PO-g': 1, 'PO-h': 1, 'PO-i': 1, 'PO-j': 1, 'PO-k': 1, 'PO-l': 1, 'PO-m': 1, 'PO-n': 1, 'PO-o': 0}, 'CO3': {'PO-a': 1, 'PO-b': 1, 'PO-c': 1, 'PO-d': 1, 'PO-e': 1, 'PO-f': 1, 'PO-g': 1, 'PO-h': 0, 'PO-i': 1, 'PO-j': 0, 'PO-k': 1, 'PO-l': 1, 'PO-m': 0, 'PO-n': 0, 'PO-o': 0}, 'CO4': {'PO-a': 1, 'PO-b': 1, 'PO-c': 1, 'PO-d': 1, 'PO-e': 1, 'PO-f': 1, 'PO-g': 1, 'PO-h': 1, 'PO-i': 1, 'PO-j': 1, 'PO-k': 1, 'PO-l': 1, 'PO-m': 1, 'PO-n': 1, 'PO-o': 0}}
